In [ ]:
# Version1_Subset_Sampling_FDR
#  << 데이터 가용성 확보를 위한 경로 리스트 구축 및 데이터셋 규격화 >>
# AI 모델링 및 학습 정확도를 위해 클래스별 500 -> 1000개로 확장 추출
import os
import random

def get_sampled_files(directory, sample_size=1000):
    valid_extensions = ('.jpg', '.jpeg', '.png', '.bmp')
    all_files = []
    for root, dirs, files in os.walk(directory):
        for file in files:
            if file.lower().endswith(valid_extensions):
                all_files.append(os.path.join(root, file))
    
    if len(all_files) > sample_size:
        sampled_files = random.sample(all_files, sample_size)
    else:
        sampled_files = all_files
    return sampled_files

FIGHTER_DIR1 = '/kaggle/input/datasets/jrmymimran/fighterjets'
FIGHTER_DIR2 = '/kaggle/input/datasets/kadirkrtls/tez-set-v1/Veri-Kumesi_V1-100/Train'
DRONE_DIR = '/kaggle/input/datasets/dasmehdixtr/drone-dataset-uav'
ROCKET_DIR = '/kaggle/input/datasets/gatewayadam/aerospace-images/aerospace_images/rockets'
ETC_DIR = '/kaggle/input/datasets/muhammadsaoodsarwar/drone-vs-bird/dataset/bird'

# 데이터 추출 및 병합 (전투기 1,000장 확보)
f_part1 = get_sampled_files(FIGHTER_DIR1, 500)
f_part2 = get_sampled_files(FIGHTER_DIR2, 500)

fighter_1000 = f_part1 + f_part2
drone_1000 = get_sampled_files(DRONE_DIR, 1000) # 드론 
rocket_1000 = get_sampled_files(ROCKET_DIR, 1000) # 로켓(Missile 대체)
etc_1000 = get_sampled_files(ETC_DIR, 1000) # 오사격 방지용 Dataset

print(f"추출 완료: 전투기({len(fighter_1000)}), 드론({len(drone_1000)}), 로켓({len(rocket_1000)}), 오사격 방지 데이터({len(etc_1000)})")

In [1]:
# Version2_Geometric_Safety_Pipeline 
# <<원본 이미지의 기하학적 형상 보존(Zero-Padding) 및 데이터 무결성 검증(Fail-Safe)이 통합된 딥러닝 전처리 파이프라인>>
import torch
from torch.utils.data import Dataset
from torchvision import transforms
from PIL import Image
import os

# 클래스 [1] : 원본 이미지 형상을 보존하며 추론 엔진용 텐서로 변환하는 데이터 공급 클래스
class AeroObjectDataset(Dataset):
    def __init__(self, file_paths, labels, img_size=224, is_train=False):
        self.file_paths = file_paths
        self.labels = labels
        self.img_size = img_size
        self.is_train = is_train
        
        if self.is_train: # [학습용] => 데이터 증강 및 텐서 변환 파이프라인
            self.transform = transforms.Compose([
                transforms.RandomHorizontalFlip(),
                # 50% 확률로 수평 반전
                transforms.RandomRotation(15),
                # 요격 각도 변화 모사 -> 각도를 너무 많이 잡으면 원본과 동 떨어진 오버피팅 우려해 15도로 초기화 
                transforms.ColorJitter(brightness=0.2, contrast=0.2),
                # 밝기/대비 변화 모사 -> 1.0으로 잡아버리면 원본이미지 훼손이 있어 학습 퀄리티 저해할 것 대비 0.2로 초기화
                transforms.GaussianBlur(kernel_size=(3, 5), sigma=(0.1, 1.0)),
                # kernel_size(눈의 시야) -> 작은 시야로 세밀하게 볼 3부터 큰 시야로 넓게도 보도록 5까지로 잡음
                # sigma(번짐의 강도) -> 블러가 거의 없는 0.1에서 강력한 번짐의 1.0까지로 잡음.
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
                # Tesnor : 3차원(층, 줄, 칸) 공간 주소에 배정된 RRB 색상값들을 엔진의 표준 규격에 맞춰 교정
                # Resnet-18의 평균 값 -> [0.485, 0.456, 0.406]
                # Resnet-18의 평균 표준편차 -> [0.229, 0.224, 0.225]
            ])
        else: # [시험용] =>증강 없이 정규화만 수행
            self.transform = transforms.Compose([
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
            ])

    # [ 종횡비 유지를 위한 중앙 정렬 제로 패딩 ]
    def _zero_padding(self, image):
        w, h = image.size
        max_dim = max(w, h)

        # 검은색(0,0,0) 정사각형 배경을 생성하고 원본 이미지를 중앙에 배치하여 종횡비 보존
        new_image = Image.new('RGB', (max_dim, max_dim), (0, 0, 0))

        # 원본 이미지를 캔버스 중앙에 배치
        new_image.paste(image, ((max_dim - w) // 2, (max_dim - h) // 2))

        # 최종 모델 입력 사이즈(224x224)로 리사이징
        # 이미지 작으면 확대 / 이미지 크면 축소 -> 기존에도 정사각형 형태라 원본에 훼손이 없음
        return new_image.resize((self.img_size, self.img_size))

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        img_path = self.file_paths[idx]
        # Fail-safe: 비정상 파일 유입 시 예외 처리 및 차단
        try:
            image = Image.open(img_path).convert('RGB') # 색상 규격을 빨/초/파 로 통일함
        except Exception:
            # 파일 손상의 기준 : 파일이 아예 안열림(헤더 손상) or 이미지 데이터가 중간에 끊긴 경우
            random_idx = random.randint(0, len(self.file_paths) - 1)
            return self.__getitem__(random_idx) # 다른 정상 샘플로 대체하여 반환
            
        padded_image = self._zero_padding(image) # 제로 패딩 적용 
        return self.transform(padded_image), torch.tensor(self.labels[idx], dtype=torch.long) # 텐서 변환 및 데이터 증강

# 클래스 [2] : 데이터 무결성을 검증하고 손상된 샘플을 데이터셋에서 영구 제외하는 모듈
class FailSafeValidator:
    def __init__(self, file_paths, labels):
        self.file_paths = file_paths
        self.labels = labels

        # 이미지가 정상일 경우 경로와 라벨(정답지) 묶어서 저장함
        self.clean_paths = []
        self.clean_labels = []

    # 전체 리스트를 순회하며 실제 열리는 이미지인 경우 clean_paths와 clean_labels에 담음
    def filter_bad_images(self):
        print(f"[*] 데이터 무결성 검사 시작 (대상: {len(self.file_paths)}개)...")
        removed_count = 0
        for path, label in zip(self.file_paths, self.labels):
            try:
                # [ 실제로 이미지가 열리고 RGB 변환이 가능한지 확인 ]
                # 자원 사용할 때 with 사용하며, 사용 끝나면 자동으로 닫음
                with Image.open(path) as img:
                    img.verify() # 파일 구조 1차 검증 (조건에 맞는 확장자인지 확인)
                with Image.open(path) as img:
                    img.load() # 실제 픽셀 데이터 손상 여부 2차 검증 (내부 데이터가 결함없는지 검증)

                # 검증 통과 시 '깨끗한 리스트'에 추가
                self.clean_paths.append(path)
                self.clean_labels.append(label)
                
            except Exception:
                removed_count += 1
                continue
                
        print(f"검사 완료 : {removed_count}개의 불량 샘플 제거됨.")
        return self.clean_paths, self.clean_labels

In [ ]:
# Version3_ResNet18_Architecture_Trainer
# << 항공 객체 식별용 다차원 특징 추출 엔진(ResNet-18) 설계 및 지도학습 최적화 모듈 >>
import torch
import torch.nn as nn
import torchvision.models as models
import torch.optim as optim

# 클래스 [3] : ResNet-18을 활용하여 추출한 데이터셋의 다차원 특징을 추출하는 신경망 클래스
class AeroObjectClassifier(nn.Module): # nn.Module 상속 -> 파이토치에서 모든 신경망 모델이 가져야할 기본 뼈대를 받음
    def __init__(self, num_classes=4, pretrained=True):
        super().__init__()
        # ResNet-18을 선택한 이유 
        # -> 이미지의 선,면 및 구체적인 형태를 추출하는데 매우 탁월한 성능을 가짐
        # -> 4000개라는 소량의 데이터에 적합하다고 판단
        weights = models.ResNet18_Weights.DEFAULT if pretrained else None
        # [참일 때의 값] if [조건문] else [거짓일 때의 값]

        # 모델 탑재
        self.backbone = models.resnet18(weights=weights)

        # fc -> 이미지 특징들을 하나로 모아 최종적인 결과를 도출하는 출력층
        # in_features 통해 모델에 적합한 규격에 맞춤 (3개의 데이터셋(클래스))
        in_features = self.backbone.fc.in_features

        # 입력 512(default)개 -> 출력 3개(전투기, 드론, 로켓)
        self.backbone.fc = nn.Linear(in_features, num_classes)
        
        # ONNX 추출 시에만 확률을 반환하도록 제어하는 스위치
        self.export_mode = False
        
    def forward(self, x):
        logits = self.backbone(x)
        # 1. 형상 보존형 전처리(Zero-Padding)를 마친 텐서 x를 입력받음 
        # 2. CNN 필터를 통해 객체의 기하학적 특징과 실루엣을 추출하여 연산 수행 
        # 3. 각 클래스(전투기, 드론, 로켓)에 대한 예측 점수(Logits)가 담긴 텐서 반환
        
        if self.export_mode:
            return torch.sigmoid(logits)
            # Zero-Sum 형태인 softmax는 실제 실행간에 부적합하다 판단
            # => 독립적 평가 형태인 sigmoid로 변경
        return logits

# 클래스 [4] : 모델을 학습시키고 가중치를 추출하는 클래스
class ModelTrainer:
    def __init__(self, model: nn.Module, device: str, learning_rate=0.001):
        self.model = model
        self.device = torch.device(device)
        self.model.to(self.device)

        # 오발사 방지를 위한 엄격한 지도학습(CrossEntropyLoss) 채택
        self.criterion = nn.BCEWithLogitsLoss()
        self.optimizer = optim.Adam(self.model.parameters(), lr=learning_rate)

    def train_epoch(self, train_loader):
        self.model.train() # 학습 모드 활성화 
        running_loss = 0.0 # 하나의 epoch가 처리되면 다시 0으로 초기화
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(self.device), labels.to(self.device)

            # 국방 시스템의 오발사 방지를 위해, 시행착오 기반의 강화학습(RL) 대신 판독 신뢰성이 확보된 지도학습(SL) 채택
            self.optimizer.zero_grad()
            outputs = self.model(inputs)
            
            # 4개 클래스 One-hot 변환
            labels_one_hot = torch.nn.functional.one_hot(labels, num_classes=4).float()
            
            loss = self.criterion(outputs, labels_one_hot)
            loss.backward()
            self.optimizer.step()
            
            # 순수 오차 수치만 누적함 (학습용)
            running_loss += loss.item()

        # Epoch의 최종 평균 오차를 계산해서 반환하는 동작
        return running_loss / len(train_loader)

    def evaluate(self, val_loader):
        self.model.eval() # 검증 모드 (가중치 업데이트 중단)
        val_loss = 0.0 # 오차 합계
        correct, total = 0, 0 # 정확히 맞춘 개수, 전체 문제 개수 누적합
        with torch.no_grad():
            for inputs, labels in val_loader:
                # val_loader가 32(batch)장의 문제지(inputs)와 정답지(labels)
                
                inputs, labels = inputs.to(self.device), labels.to(self.device)
                outputs = self.model(inputs)
                labels_one_hot = torch.nn.functional.one_hot(labels, num_classes=4).float()
                loss = self.criterion(outputs, labels_one_hot) # 32장 묶음에 대한 오차값(Tensor)
                val_loss += loss.item() # 텐서의 껍데기 벗기고 알맹이 숫자만 뽑아서 더함
                _, predicted = torch.max(outputs.data, 1)
                # 각 이미지당 '가장 높은 점수'와 그 '점수의 인덱스' 두 가지를 반환함
                # torch.max라는 함수는 반드시 두개의 세트를 반환하도록 되어있지만
                # -> 점수자체(_)는 필요X, 몇 번 클래스 찍었는지 인덱스(predicted)가 필요
                
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
                # 모델이 찍은 답(predicted)와 실제 정답(labels)일치하는 지를 비교
                # => [True, False, True ... ] 형태의 텐서가 나옴
                # .item을 통해 맞춘 개수(정수 형태)만 correct에 저장됨
        
        return val_loss / len(val_loader), 100 * correct / total

In [ ]:
# Version4_ReasoningGuard_Core_Engine_Generation
# << Reasoning Guard 실전 파이프라인 총괄 컨트롤러 및 실행 모듈 >>
# Scikit-learn(딥러닝X, 고전적이지만 회귀, 클러스트링 같은 강력한 AL 포함 -> 빠름) 등이 존재
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
import time

# 클래스 [5] : 클래스 [1],[2],[3],[4] 포함한 메인 학습&추출 클래스
class ReasoningGuardPipeline:

    # 파이프라인 가동되기 전, 필요한 기본 설정값과 자원을 할당받음
    def __init__(self, all_paths, all_labels, batch_size=32, num_epochs=20): # epochs: 10 -> 20
        self.all_paths, self.all_labels = all_paths, all_labels
        # Where (paths -> 모든 데이터들의 주소) 
        # What (labels -> 정답 번호표)
        
        self.batch_size, self.num_epochs = batch_size, num_epochs
        # How much at once (size -> 동시처리할 데이터 묶음 단위)
        # How many times (epoch -> 전체 복습 횟수)
        
        self.device = "cuda" if torch.cuda.is_available() else "cpu"

        # 내부에서 사용할 객체들 초기화 대기
        self.train_loader, self.val_loader = None, None
        self.model, self.trainer = None, None

    # 전처리 및 데이터 로드 단계
    def prepare_data(self):
        print("\n[Phase 1] 데이터 준비 및 Fail-safe 검증 가동")

        # ★ FailSafeValidator 클래스[2] 호출 => 깨지거나 손상된 이미지 원천 차단
        validator = FailSafeValidator(self.all_paths, self.all_labels)
        clean_paths, clean_labels = validator.filter_bad_images()

        # 전체 데이터를 학습용과 검증용 (8:2)로 나눔
        train_paths, val_paths, train_labels, val_labels = train_test_split(
            clean_paths, clean_labels, test_size=0.2, stratify=clean_labels, random_state=42
        )

        # ★ AeroObjectDataset 클래스[1] 호출
        # Train (80%) / Validation (20%) Dataset 생성
        # val_dataset은 is_train=False를 통해 데이터 증강 가동하지 않음 -> 검증 및 실전 모드
        train_dataset = AeroObjectDataset(train_paths, train_labels, img_size=224, is_train=True)
        val_dataset = AeroObjectDataset(val_paths, val_labels, img_size=224, is_train=False)

        # DataLoader 설정
        self.train_loader = DataLoader(train_dataset, batch_size=self.batch_size, shuffle=True, num_workers=2, drop_last=True)
        self.val_loader = DataLoader(val_dataset, batch_size=self.batch_size, shuffle=False, num_workers=2)

    # 클래스 [3],[4] 활용하여 실질적인 학습을 시키는 함수
    def execute_training(self):
        print(f"\n[Phase 2] 전술 객체 식별 AI 훈련 개시 (총 {self.num_epochs} Epochs)")

        # 연산 장치 및 추론 엔진 초기화
        self.model = AeroObjectClassifier(num_classes=4, pretrained=True)
        self.trainer = ModelTrainer(model=self.model, device=self.device, learning_rate=0.001)

        # 본격적인 학습 및 검증 루프
        for epoch in range(self.num_epochs):
            
            # [훈련 모드] -> 평균 오차 점수 저장 (낮을수록 좋음)
            train_loss = self.trainer.train_epoch(self.train_loader)

            # [검증 모드] -> 시험의 오차, 정답률 저장
            val_loss, val_acc = self.trainer.evaluate(self.val_loader)

            # 1번의 epoch 수행하는데 걸린시간 도출
            print(f"Epoch [{epoch+1}/{self.num_epochs}] | Val Acc: {val_acc:.2f}%")

    def export_to_onnx(self):
        print("\n[Phase 3] ONNX 엔진 추출")
        
        self.model.eval()

        # 모델 내부 스위치를 킴
        self.model.export_mode = True

         # torch.randn(한번에 처리할 이미지 개수, RGB컬러이미지, 이미지의 가로세로 해상도)
        dummy_input = torch.randn(1, 3, 224, 224).to(self.device)

        torch.onnx.export(self.model, dummy_input, "reasoning_guard_engine.onnx", opset_version=18)

# ============================================
# 실제 컨트롤러 가동 (모든 파이프라인이 실행)
# ============================================
if __name__ == "__main__": # 사용자가 직접 이 파일을 실행했을때만 코드 동작하도록 제안함
    all_paths = fighter_1000 + drone_1000 + rocket_1000 + etc_1000
    all_labels = [0]*1000 + [1]*1000 + [2]*1000 + [3]*1000

    # 파일 경로와 라벨 리스트 병합 (상단에서 이미 선언된 데이터 사용)
    pipeline = ReasoningGuardPipeline(all_paths, all_labels)
    pipeline.prepare_data() # ★ FailSafeValidator, AeroObjectDataset 클래스 1,2 호출
    pipeline.execute_training() # ★ AeroObjectClassifier, ModelTrainer 클래스 3,4 호출
    pipeline.export_to_onnx() # onnx로 추출

    # C++ 연동 전, 파이썬 환경에서 3단계 전술 대응 로직 시뮬레이션
    # (본 엔진 로직에 영향을 주지 않는 독립 검증 코드)
    print("\n[Phase 4] 3단계 전술 대응(SAFE/CAUTION/DANGER) 시뮬레이션")
    pipeline.model.eval()
    
    with torch.no_grad():
        # 검증 데이터에서 한 배치(32개)를 가져옴
        inputs, _ = next(iter(pipeline.val_loader))
        probs_batch = torch.sigmoid(pipeline.model(inputs.to(pipeline.device)))
        # 확률값 추출(sigmoid 결과)
        
        print("-" * 85)

         # [Tactical Logic] 변수에 판정 결과와 메시지를 먼저 할당함
        for i in range(5): # 상위 5개 샘플 테스트
            p = probs_batch[i].cpu().tolist()
            f, d, r, bird = p[0], p[1], p[2], p[3]
            max_threat = max(f, d, r)
            
            if max_threat > 0.90 and bird < 0.50:
                status, msg = "DANGER (Shoot)", "위협 확정 => 즉각 대응"
            elif max_threat < 0.40 or bird >= 0.50:
                status, msg = "SAFE (Hold)", "비위협(새) 또는 불확실 => 보류"
            else:
                status, msg = "CAUTION (Override)", "판정 경합 => 수동 승인 필요"

            # 최종 출력
            print(f"Sample {i+1} | [F:{f:.2f} D:{d:.2f} R:{r:.2f} Bird:{bird:.2f}] -> {status}")
        print("-" * 85)